In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [4]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

In [5]:
# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdfs/")

Found 4 PDF files to process

Processing: DMDWactivity20.pdf
  ✓ Loaded 4 pages

Processing: group-9 lab_report.pdf
  ✓ Loaded 30 pages

Processing: ml_activity11.pdf
  ✓ Loaded 5 pages

Processing: Time To Explore- [A-15].pdf
  ✓ Loaded 5 pages

Total documents loaded: 44


In [6]:
all_pdf_documents

[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-03-28T16:27:13+00:00', 'title': 'A [20]', 'moddate': '2026-03-28T16:27:13+00:00', 'keywords': 'DAHAeA4hOQc,BAG5ByuoPkI,0', 'author': 'Shresth Kumar Jha', 'source': '..\\data\\pdfs\\DMDWactivity20.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'DMDWactivity20.pdf', 'file_type': 'pdf'}, page_content='Ensemble Learning\n1. What do you mean by Ensemble Learning? Investigate common \nensemble learning methods.\nEnsemble Learning is a machine learning technique where multiple models (called weak \nlearners) are combined to improve overall prediction accuracy. Instead of relying on a single \nmodel, ensemble learning aggregates predictions from several models to produce a better and \nmore stable result. It helps reduce overfitting, bias, and variance.\nThe main idea is:\n Many weak models → combined → strong model\nCommon Ensemble Learning Methods\n1. Bagging (Bootstrap Aggregating)\nBagg

In [7]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=150,chunk_overlap=50):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:50]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [8]:
chunks = split_documents(all_pdf_documents, chunk_size=150, chunk_overlap=50)
chunks

Split 44 documents into 246 chunks

Example chunk:
Content: Ensemble Learning
1. What do you mean by Ensemble ...
Metadata: {'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-03-28T16:27:13+00:00', 'title': 'A [20]', 'moddate': '2026-03-28T16:27:13+00:00', 'keywords': 'DAHAeA4hOQc,BAG5ByuoPkI,0', 'author': 'Shresth Kumar Jha', 'source': '..\\data\\pdfs\\DMDWactivity20.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'DMDWactivity20.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-03-28T16:27:13+00:00', 'title': 'A [20]', 'moddate': '2026-03-28T16:27:13+00:00', 'keywords': 'DAHAeA4hOQc,BAG5ByuoPkI,0', 'author': 'Shresth Kumar Jha', 'source': '..\\data\\pdfs\\DMDWactivity20.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'DMDWactivity20.pdf', 'file_type': 'pdf'}, page_content='Ensemble Learning\n1. What do you mean by Ensemble Learning? Investigate common \nensemble learning methods.'),
 Document(metadata={'producer': 'Canva', 'creator': 'Canva', 'creationdate': '2026-03-28T16:27:13+00:00', 'title': 'A [20]', 'moddate': '2026-03-28T16:27:13+00:00', 'keywords': 'DAHAeA4hOQc,BAG5ByuoPkI,0', 'author': 'Shresth Kumar Jha', 'source': '..\\data\\pdfs\\DMDWactivity20.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'DMDWactivity20.pdf', 'file_type': 'pdf'}, page_content='ensemble learning methods.\nEnsemble Learning is a machine learning technique 

embedding And vectorStoreDB

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

d:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
